# SIITec Hackathon 2569 · รอบคัดเลือก
### Vision-Language AI for Smart Factory — หาตำหนิชิ้นงานด้วยการพิมพ์คำค้น

**วิธีเริ่ม:** เมนู *Runtime ▸ Change runtime type ▸ T4 GPU* แล้ว *Runtime ▸ Run all*

1. **คำค้น** — พิมพ์คำภาษาอังกฤษบอก OWLv2 ว่าตำหนิแต่ละแบบหน้าตาเป็นอย่างไร
2. **`my_filter`** — เขียน Python กรองหรือแก้กรอบต่อเอง
3. **`my_draw`** — วาดภาพผลลัพธ์ด้วย OpenCV ให้สวยตามสไตล์ทีม
4. **ส่งงาน** — ส่งออกไฟล์ `.json` แล้วส่งพร้อมวิดีโอนำเสนอ

In [ ]:
#@title ① เตรียมเครื่อง — กด ▶ ครั้งเดียว (ประมาณ 2 นาที)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/SIITec-Hackathon-2026/round1/main/"
for f in ["siitec_owl.py", "siitec_ui.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)
import siitec_ui
app = siitec_ui.App()

## ② `my_filter` — เลือกกรอบด้วย Python
ตั้งเกณฑ์ความมั่นใจ และเขียนเงื่อนไขเพิ่มได้ตามใจ — แก้แล้วกด ▶ ที่เซลล์ แล้วกด **▶ ทดสอบ** บนหน้าจอ
ใช้ได้: Python · NumPy · OpenCV — **ไม่อนุญาต** ให้เทรนหรือใช้โมเดลอื่น

In [ ]:
import numpy as np
import cv2

# เกณฑ์ความมั่นใจของแต่ละแบบ (0 ถึง 1) — กรอบที่คะแนนต่ำกว่านี้จะถูกตัดทิ้ง
THRESHOLD = {
    "bottle_broken_large":  0.03,
    "bottle_broken_small":  0.03,
    "bottle_contamination": 0.03,
    "metal_nut_bent":       0.03,
    "metal_nut_color":      0.03,
    "metal_nut_flip":       0.03,
    "metal_nut_scratch":    0.03,
    "screw_defect":         0.03,
}


def my_filter(image, detections):
    """
    image      : ภาพ 1 ภาพ (numpy array แบบ RGB ขนาด สูง x กว้าง x 3)
    detections : กรอบทั้งหมดที่โมเดลเสนอมา
                 [{"label": "metal_nut_bent", "score": 0.31, "box": [x1, y1, x2, y2]}, ...]
    คืนค่า      : รายการกรอบที่อยากเก็บไว้
    """
    h, w = image.shape[:2]
    keep = []
    for d in detections:
        if d["score"] < THRESHOLD[d["label"]]:
            continue
        # ไอเดียให้ลอง:
        # - ตัดกรอบที่เล็กหรือใหญ่ผิดปกติ   x1, y1, x2, y2 = d["box"]
        # - ตัดกรอบที่อยู่นอกตัวชิ้นงาน (หาตัวชิ้นงานด้วย cv2.threshold / cv2.findContours)
        # - ภาพละไม่เกินกี่กรอบ หรือเก็บเฉพาะแบบที่คะแนนสูงสุด
        keep.append(d)
    return keep

## ③ `my_draw` — วาดผลลัพธ์ด้วย OpenCV
ภาพในแท็บ **ผลลัพธ์** วาดด้วยฟังก์ชันนี้ ลองเปลี่ยนสี เส้น ป้าย หรือใส่ลูกเล่นให้เหมาะกับวิดีโอของทีม
(`cv2.putText` เขียนได้เฉพาะภาษาอังกฤษ)

In [ ]:
import cv2


def my_draw(image, found, answers):
    """
    วาดผลลัพธ์ลงบนภาพด้วย OpenCV แล้วคืนภาพกลับไป (ภาพเป็น RGB)
    found   : กรอบที่หาเจอ (หลัง my_filter)   answers : กรอบคำตอบ (มีเฉพาะชุดฝึก)
    """
    img = image.copy()
    t = max(2, img.shape[0] // 300)                     # ความหนาเส้นตามขนาดภาพ

    for a in answers:                                   # คำตอบ: เส้นขาวทับเส้นเขียว
        x1, y1, x2, y2 = map(int, a["box"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), t + 2)
        cv2.rectangle(img, (x1, y1), (x2, y2), (40, 200, 120), t)

    for d in found:                                     # ที่หาเจอ: กรอบส้ม + ชื่อ + คะแนน
        x1, y1, x2, y2 = map(int, d["box"])
        color = (255, 140, 0)                           # (R, G, B)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, t + 1)
        text = d["label"].split("_")[-1] + f' {d["score"]:.2f}'
        cv2.putText(img, text, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                    t / 3, color, max(1, t // 2), cv2.LINE_AA)
    return img

## ④ หน้าจอหลัก

In [ ]:
#@title ④ เปิดหน้าจอ
app.show()